In [ ]:
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas import evaluate
import json
from dotenv import load_dotenv
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
import os
from ragas.metrics import (
    faithfulness,
    answer_correctness
)
import numpy as np
load_dotenv(override=True)



from ragas.llms import LangchainLLMWrapper

# Choose the appropriate import based on your API:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
import re

def normalize(text):
    # collapse multiple spaces/newlines into single spaces
    return re.sub(r'\s+', ' ', text).strip()

def context_preciusion_recall2(ground_truth_docs, retrieved_docs):

    ground_truth_set = set(normalize(doc) for doc in ground_truth_docs)
    retrieved_set = set(normalize(doc) for doc in retrieved_docs)

    true_positives = len(ground_truth_set.intersection(retrieved_set))


    false_positives = len(retrieved_set - ground_truth_set)
    false_negatives = len(ground_truth_set - retrieved_set)

    precision = (
        true_positives / (true_positives + false_positives)
        if (true_positives + false_positives) > 0 else 0
    )

    recall = (
        true_positives / (true_positives + false_negatives)
        if (true_positives + false_negatives) > 0 else 0
    )

    return precision, recall
# Initialize with Google AI Studio

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4.1-nano"
))

paths = [
            
            'hotpot_gemini-2.5-flash-lite_5_chunks-embedding_01-rag.json'
]

for path in paths:

    with open('../rag_answer_data/'+ path) as f:
        data = json.load(f)
    # print(data[0])
    final_data = []
    for item in data:
        rag_context_list = item['rag_context']
        final_rag_context = [item['page_content'] for item in rag_context_list]
        final_data.append({
            'question': item['multi_hop_question'],
            'ground_truth_answer': item['multi_hop_answer'],
            'relevant_docs': item['chunks'] if 'chunks' in item else item['supporting__sentences'],
            'rag_answer': item['rag_answer'] if isinstance(item['rag_answer'], str) else item['rag_answer'][1],
            'retrieved_docs': final_rag_context,
            'type': item['type'] if 'type' in item else 'unknown',
        })





    qas = []
    for qa in final_data:
        
        
        
        qas.append(
            SingleTurnSample(
                user_input=qa['question'],
                reference=qa['ground_truth_answer'],
                response=qa['rag_answer'],
                retrieved_contexts=qa['retrieved_docs'],
                reference_contexts=qa['relevant_docs'],
            )
        )

        

    result = evaluate(
        dataset=EvaluationDataset(samples=qas),
        metrics=[answer_correctness,faithfulness],
        llm=evaluator_llm,
    )
    result_json = []
    results = result.to_pandas()
    for _, row in results.iterrows():
        user_input = row["user_input"]      
        correctness_critic = row["answer_correctness"]
        faithfulness_critic = row["faithfulness"]
        precision, recall = context_preciusion_recall2(row["reference_contexts"], row["retrieved_contexts"])

        
        result_json.append({
            "content": user_input,
            "correctness_aspect_critic": correctness_critic,
            "faithfulness_aspect_critic": faithfulness_critic,
            "context_precision": precision,
            "context_recall": recall,
        })

    # Calculate averages excluding NaN values
    correctness_values = [entry["correctness_aspect_critic"] for entry in result_json]
    faithfulness_values = [entry["faithfulness_aspect_critic"] for entry in result_json]
    precision_values = [entry["context_precision"] for entry in result_json]
    recall_values = [entry["context_recall"] for entry in result_json]

    avg_correctness = np.nanmean(correctness_values)
    avg_faithfulness = np.nanmean(faithfulness_values)
    avg_precision = np.nanmean(precision_values)
    avg_recall = np.nanmean(recall_values)

    result_json = {
        "file_path": path,
        "average_scores": {
            "correctness_aspect_critic": avg_correctness,
            "faithfulness_aspect_critic": avg_faithfulness,
            "context_precision": avg_precision,
            "context_recall": avg_recall,
        },
        "detailed_results": result_json
    }
    with open('./evaluation_results_'+ path, 'w') as f:
        json.dump(result_json, f, indent=4)